# Opinion Survey Analysis

Author: Jennifer Le  
Date: 12/6/24

The purpose of this analysis is to   
1. Examine how opinions on topics like the economy has changed from 2020 to 2024.  
2. Discover groups that are affected the most when opinions about the econmy decline

The data used for this project is from the National Public Opinion Reference surveys (NPORS) conducted by the Pew Reserach Center. 

This survey includes questions on demographics like income, education, and race. It also includes questions about political views, social media use, opinions on the economy, piety and dedication to religious beliefs, and more. Click on the link below to see the 2022 Questionnaire. Navigate to other folders to see questionnaires for other years. 

<a href="https://github.com/JenniferMLe/Opinion-Survey-Analysis/blob/main/Datasets/NPORS-2022/Questionnaire_22.pdf" target="_blank">See questions asked for 2022 study</a>  
<a href="https://www.pewresearch.org/methods/fact-sheet/national-public-opinion-reference-survey-npors/" target="_blank">More information on NPORS</a>  
<a href="https://jennifermle.github.io/Opinion-Survey-Analysis/" 
target="_blank">Click to see graphs (if viewing main.ipynb)</a>

The concept for this project stems from an assignment from Prof. Elena Zheleva's Intro to Data Science class (CS 418), spring 2024.

## Helper Functions

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

def write_to_file(df,file_name='Result_Datasets/result.csv'):
    df.to_csv(file_name, index=False)

# print a list of distinct values from a column in a dataframe
# helps with data cleaning
def get_distinct_values(df, columns, sort_by):
    vals = df[columns].sort_values(by=sort_by).drop_duplicates()
    # print to csv file to see all values since output may be cut if too long
    write_to_file(vals)

def get_count(cols):
    # as_index=False doesn't make group labels the index
    df = df_combined.groupby(cols, as_index=False)['RESPID'].count()
    df = df.rename(columns={'RESPID':'Count'})
    for col in cols:
        df = df[df[col] != 'N/A']
    write_to_file(df)
    return df

def get_count_weighted(cols):
    # as_index=False doesn't make group labels the index
    df = df_combined.groupby(cols, as_index=False)['WEIGHT'].sum().round(0)
    df = df.rename(columns={'WEIGHT':'Count'})
    for col in cols:
        df = df[df[col] != 'N/A']
    write_to_file(df)
    return df

category_orders = {
    'AGEGRP2':['18-24','25-39','40-59','60-79','80+'],

    'INCOMEGRP':['< $40K','$40-70K','$70-100K','$100K+'],

    'ECON1MOD':['Poor','Only fair','Good','Excellent'],

    'ECON1BMOD':['Better','About the same','Worse'],
    
    'EDUCATION':[
        "No schooling completed",
        "Some High School",
        "High School",
        "Some College",
        "Associate's Degree",
        "Bachelor's Degree",
        "Master's Degree or Higher"
    ]
}

income_map2 = {
    'Under $40k':"#3d005e",
    '$40-70K':"#8a03d5",
    '$70-100K':"#bd45ff",
    '$100K+':"#d995ff"
}


## Importing Data

In [23]:
# Save each .sav as a data frame
df_20 = pd.read_spss("Datasets/NPORS-2020/dataset.sav")
df_21 = pd.read_spss("Datasets/NPORS-2021/dataset.sav")
df_22 = pd.read_spss("Datasets/NPORS-2022/dataset.sav")
df_23 = pd.read_spss("Datasets/NPORS-2023/dataset.sav")
df_24 = pd.read_spss("Datasets/NPORS-2024/dataset.sav")
df_25 = pd.read_spss("Datasets/NPORS-2025/dataset.sav")

# convert .sav to .csv
write_to_file(df_20,"Datasets/NPORS-2020/dataset_20.csv")
write_to_file(df_21,"Datasets/NPORS-2021/dataset_21.csv")
write_to_file(df_22,"Datasets/NPORS-2022/dataset_22.csv")
write_to_file(df_23,"Datasets/NPORS-2023/dataset_23.csv")
write_to_file(df_24,"Datasets/NPORS-2024/dataset_24.csv")
write_to_file(df_25,"Datasets/NPORS-2025/dataset_25.csv")

## Combining Data

In [24]:
# stop number of columns at the end that don't have the year in the column name
def remove_year_from_column_name(dataset, stop):
    list_columns = list(dataset.columns)

    for i in range(0,len(list_columns)-stop):
        list_columns[i] = list_columns[i][:-5]

    dataset.columns = list_columns

# Remove the year from column name to keep naming consistant
remove_year_from_column_name(df_20, 5)
remove_year_from_column_name(df_21, 2)

# ensure columns recording the same data have the same name
# so there aren't duplicate columns when appending datasets
df_20 = df_20.rename(columns={
    'SEXASK':'GENDER', 
    'EDUC_ACS':'EDUCATION'
})

df_23 = df_23.rename(columns={
    'BASEWT':'BASEWEIGHT', 
    'INC_SDT1':'INCOME'
})

df_24 = df_24.rename(columns={
    'BASEWT':'BASEWEIGHT', 
    'INC_SDT1': 'INCOME',
    'SMUSEa' : 'SMUSE_a','SMUSEd' : 'SMUSE_d','SMUSEg' : 'SMUSE_g','SMUSEj' : 'SMUSE_j',
    'SMUSEb' : 'SMUSE_b','SMUSEe' : 'SMUSE_e','SMUSEh' : 'SMUSE_h','SMUSEk' : 'SMUSE_k',
    'SMUSEc' : 'SMUSE_c','SMUSEa' : 'SMUSE_f','SMUSEi' : 'SMUSE_i'
})

df_25 = df_25.rename(columns={
    'BASEWT':'BASEWEIGHT', 
    'INC_SDT1': 'INCOME'
})

# add year columns to all datasets
df_20['YEAR'] = 2020
df_21['YEAR'] = 2021
df_22['YEAR'] = 2022
df_23['YEAR'] = 2023
df_24['YEAR'] = 2024
df_25['YEAR'] = 2025

# Combine (append) all datasets together
df_combined = pd.concat([df_20, df_21, df_22, df_23, df_24, df_25])
print(df_combined.shape)

# Keep relevent columns only
df_combined = df_combined[[
    'RESPID','YEAR','AGE','AGEGRP','GENDER','RACECMB', # basic demographics
    'INCOME','EDUCATION', 'RELIG', 'PARTY', # other useful demographics 
    'RELIMP', 'PRAY', 'MARITAL', # other features
    'SMUSE_a', 'SMUSE_b', 'SMUSE_c', 'SMUSE_d', 'SMUSE_e', 'SMUSE_f', # social media use
    'SMUSE_g','SMUSE_h','SMUSE_i','SMUSE_j','SMUSE_k',
    'ECON1MOD', 'ECON1BMOD', # we want to study how this changes over time
    'BASEWEIGHT', 'WEIGHT' 
]]


(28469, 123)


## Cleaning Data

In [25]:
# rename columns
df_combined = df_combined.rename(columns={
    'SMUSE_a':'FACEBOOK',
    'SMUSE_b':'YOUTUBE',
    'SMUSE_c':'TWITTER',
    'SMUSE_d':'INSTAGRAM',
    'SMUSE_e':'SNAPCHAT',
    'SMUSE_f':'WHATSAPP',
    'SMUSE_g':'LINKEDIN',
    'SMUSE_h':'PINTEREST',
    'SMUSE_i':'TIKTOK',
    'SMUSE_j':'BEREAL',
    'SMUSE_k':'REDDIT',
    'RACECMB':'RACE'
})

'''
replace values for consistency 
'''
# change n/a to -1 so we can convert age to float
df_combined["AGE"] = df_combined["AGE"].replace({
    "n/a":"-1",
    "98+":"98",
    "":"-1",
    'Refused':"-1"
})

# change column type
df_combined["AGE"] = df_combined["AGE"].astype(float)


df_combined = df_combined.replace({
    r'.*Refused.*':'N/A', 
    r'.*Something else.*':'Other',
    'No, don\'t use this':'No use',
    'No, don’t use this':'No use',
    "Yes, use this":'Yes use',
},regex=True)

df_combined['GENDER'] = df_combined['GENDER'].replace({
    "A man":"Male",
    "A woman":"Female",
    "In some other way":"Other"
})

df_combined["RACE"] = df_combined["RACE"].replace({
    r'.*Asian.*':'Asian',
    r'.*Black.*':'Black',
    r'.*other.*':'Other',
    'Mixed race':'Mixed Race'
},regex=True)

df_combined['ECON1MOD'] = df_combined['ECON1MOD'].replace('Only Fair','Only fair')

df_combined["INCOME"] = df_combined["INCOME"].replace({
    r' to less than ':'-',
    r' or more':'+',
    r'Less than':'<',
    r',000':'K',
},regex=True)

df_combined["EDUCATION"] = df_combined["EDUCATION"].replace({
    r'.*11.*':'Some High School',
    r'.*12.*':'Some High School',
    r'.*high school.*':'High School',
    r'.*GED.*':'High School',
    r'.*college.*':'Some College',
    r'.*Associate.*':'Associate\'s Degree',
    r'.*Bachelor.*':'Bachelor\'s Degree',
    r'.*Master.*':'Master\'s Degree or Higher',
    r'.*MD.*':'Master\'s Degree or Higher',
    r'.*Doctorate.*':'Master\'s Degree or Higher'
},regex=True)

df_combined["RELIG"] = df_combined["RELIG"].replace({
    r'.*Mormon.*':'Mormon',
    r'.*Orthodox.*':'Orthodox',
    r'.*Protestant.*':'Protestant'
},regex=True)

df_combined['MARITAL'] = df_combined['MARITAL'].replace({
    "Separated":"Divorced",
    "Never been married":"Never married",
})

categorical_cols = df_combined.select_dtypes(include="category").columns
df_combined[categorical_cols] = df_combined[categorical_cols].astype(str)

write_to_file(df_combined, 'Result_Datasets/combined_dataset.csv')
print(df_combined.shape)


(28469, 28)


## Creating Calculated Columns

In [26]:
'''create calculated columns to group incomes'''
# conditions for each group
conditions = [
    df_combined['INCOME'].isin(['$10K-$20K','$20K-$30K','$30K-$40K','< $10K','< $30K']),
    df_combined['INCOME'].isin(['$40K-$50K','$50K-$60K','$60K-$70K','$50K-$70K']),
    df_combined['INCOME'].isin(['$70K-$100K','$70K-$80K','$70K-$90K','$75K-$100K','$80K-$90K','$90K-$100K']),
    df_combined['INCOME'].isin(['$100K+','$100K-$125K','$100K-$150K','$125K-$150K','$150K+'])
]
# corresponding groups for each condition
group = ['< $40K','$40-70K','$70-100K','$100K+']

# insert new column after INCOME
df_combined.insert(
    df_combined.columns.get_loc('INCOME') + 1, # position we want to insert at
    'INCOMEGRP', # name of new column
    np.select(conditions, group, default='N/A') # set value according to conditions 
)

'''create calculated columns to group incomes'''
conditions = [
    (df_combined['AGEGRP'] == '18-24') | ((18 <= df_combined['AGE']) & (df_combined['AGE'] <= 24)),
    (df_combined['AGEGRP'].isin(['25-29','30-34','35-39'])) | ((25 <= df_combined['AGE']) & (df_combined['AGE'] <= 39)),
    (df_combined['AGEGRP'].isin(['40-44','45-49','50-54','55-59'])) | ((40 <= df_combined['AGE']) & (df_combined['AGE'] <= 59)),
    (df_combined['AGEGRP'].isin(['60-64','65-69','70-74','75-79'])) | ((60 <= df_combined['AGE']) & (df_combined['AGE'] <= 79)),
    (df_combined['AGEGRP'] == '80+')| (80 <= df_combined['AGE'])
]
group = ['18-24','25-39','40-59','60-79','80+']

df_combined.insert(
    df_combined.columns.get_loc('AGEGRP') + 1, # position we want to insert at
    'AGEGRP2', # name of new column
    np.select(conditions, group, default='N/A') # set value according to conditions 
)

## Suvery Participant Demographics

In this section, we will get to know our 2020-2024 survey participants and who they consist of by examining their age, race, gender, income, and education.

In [ ]:
'''Age Distribution'''

fig_age = px.bar(
    get_count(['AGEGRP2']), 
    x = 'AGEGRP2', 
    y = 'Count',
    title = "Demographic Breakdown by Age",
    category_orders = category_orders,
    color_discrete_sequence=px.colors.sequential.Viridis
)
fig_age.update_xaxes(title="Age").show()

'''Race Distribution'''

fig_race = px.bar(
    get_count(['RACE']), 
    y='RACE', 
    x='Count',
    title='Demographic Breakdown by Race',
    text_auto='.2s', # put numbers in K format
    color_discrete_sequence=['#ffd525']
    
)
# add subtitle
fig_race.update_layout(
    annotations=[
        dict(
            x=-0.45, y=-0.20,  # Position below the chart
            text="*Participants can select more than 1 race",
            showarrow=False,  # No arrow pointing to the text
            xref="paper", yref="paper",  # Position relative to the chart
            font=dict(size=12, color="gray"),
        )
    ]
)
fig_race.update_yaxes(title="").update_layout(yaxis={'categoryorder':'total descending'}).show()

'''Income Distribution'''

fig_income = px.bar(
    get_count(['INCOMEGRP']),
    x='INCOMEGRP',
    y='Count',
    title='Demographic Breakdown by Income Group',
    text_auto='.2s',
    color_discrete_sequence=['#77be34'],
    category_orders= category_orders
)

# add subtitle
fig_income.update_layout(
    annotations=[
        dict(
            x=-0.08, y=-0.20,  # Position below the chart
            text="*Participants in the $50-75K group (665 people) are excluded",
            showarrow=False,  # No arrow pointing to the text
            xref="paper", yref="paper",  # Position relative to the chart
            font=dict(size=12, color="gray"),
        )
    ]
).update_xaxes(title="Income Group").show()

'''Education Distribution'''

fig_education = px.bar(
    get_count(['EDUCATION']),
    y = 'EDUCATION',
    x = 'Count',
    title = 'Demographic Breakdown by Education',
    color_discrete_sequence=['#063dcf'], 
    text='Count',
    category_orders=category_orders
)
fig_education.update_yaxes(title="Education").show()


## Exploring Changes in Opinions from 2020 to 2024

In [114]:
color_map = {
    'Excellent':"#0daa00",
    'Good':'#8fdc32',
    'Only fair':'#ffc500',
    'Poor':'#f6492a',
    'Better':"#8fdc32",
    'About the same':'#ffc500',
    'Worse':'#f6492a'
}
# this function displays a cluster bar graph dispalying yearly changes in a column in the combined dataframe
def examine_changes(column,weighted, title):
    if weighted:
        df_change = get_count_weighted(['YEAR',column])
    else:
        df_change = get_count(['YEAR',column])
    df_change['Percent'] = (df_change['Count'] / df_change.groupby('YEAR')['Count'].transform('sum')).round(4) * 100
    write_to_file(df_change)

    fig = px.line(
        df_change,
        x='YEAR',
        y='Percent',
        color=column,
        title=title,
        hover_data=['Count'],
        category_orders=category_orders,
        color_discrete_map=color_map
    )
    # type='category' force x-axis to be discrete categories, not continuous bins so years won't be combined
    (fig
        .update_xaxes(type='category',title_text='')
        .update_traces(line=dict(width=8))
        .update_layout(legend_title_text='')
        .show()
    )

In [115]:
examine_changes('ECON1MOD',True,'Economy Rating Percentages by Year')
examine_changes('ECON1BMOD',True,'Economy 1-Year Outlook Percentages by Year')

## Going Into the Details

Which group's percentage of negative sentiment on the economy increased the most for each year? 

In [174]:
def get_percent_increase(column,weighted=True):
    if weighted: df = get_count_weighted(['YEAR',column,'ECON1MOD'])
    else: df = get_count(['YEAR',column,'ECON1MOD'])

    conditions = [
        (df['ECON1MOD'] == 'Excellent') | (df['ECON1MOD'] == 'Good'),
        (df['ECON1MOD'] == 'Poor') | (df['ECON1MOD'] == 'Only fair')
    ]
    group = ['Positive', 'Negative']

    # create a new column that labels each rating as negative or positive
    df['SENTIMENT'] = np.select(conditions, group, default='N/A')

    # get the count for each unique year + group + sentiment
    df = df.groupby(['YEAR',column,'SENTIMENT'],as_index=False)['Count'].sum()

    # convert to wide format so negative rating counts and positive rating counts are a separate columns
    df = pd.pivot_table(df,index=['YEAR',column],columns=['SENTIMENT'],values='Count').reset_index()

    # add a column for the percent of negative ratings 
    df['PercentNegative'] = round((df['Negative'] / (df['Negative']+df['Positive']))*100,1)

    # convert to wide so each year gets its own column with percent negative as the values
    df = pd.pivot_table(df,index=column,columns='YEAR',values='PercentNegative').reset_index()
    
    # calculate the difference between the percent of negative ratings for the current and previous year
    # 
    col_names = []
    for year in df.columns[2:]:
        col_name = str(year)
        df[col_name] = round(df[year] - df[year-1],1)
        col_names.append(col_name)
    
    # exclude all columns but the group name and the percent difference columns
    df = df[[column]+col_names]
    
    # convert back to long format so we can graph by Year
    df = pd.melt(df,id_vars = [column], value_vars=col_names,var_name='YEAR',value_name='PercentNegDiff')
    write_to_file(df)
    return df

def graph_percent_increase(col,group_name):
    df = get_percent_increase(col)

    fig = px.bar(
        df,
        x='YEAR',
        y='PercentNegDiff',
        title='Difference in Negative Economy Ratings Percentages From Previous Year For ' + group_name + ' Groups',
        barmode='group',
        color=df.columns[0],
        hover_data=['YEAR'],
        category_orders=category_orders,
        color_discrete_map=color_map
    )
    (fig
        .update_yaxes(title="Difference")
        .update_xaxes(type='category')
        .update_layout(legend_title_text=group_name)
        .show())

graph_percent_increase('INCOMEGRP','Importance-of-Religion')

## Hypothesis Test
Let's see if groupos like income, age, education, or religion affects opinions about the economy.

Null Hypothesis - In 2022, age does not influence opinions on economic conditions. 

If we accept this, it means the percentage of total answers for an age group should approximately equal the percentage of total negative answers for that age group. For example, if the percentage of all responses from 18-24 year olds is 25%, the percentage of negative responses for 18-24 year olds should be around 25%. 

Alternative Hypothesis - Age does does influence opinions on economic conditions

## Groups with Largest Increase in Negative Oppinions

Maybe delete graph later

In [ ]:
groups = ['AGEGRP2','GENDER', 'RACE','INCOMEGRP', 'EDUCATION', 'RELIG', 'PARTY', 'RELIMP', 'PRAY','MARITAL',
          'YOUTUBE', 'TWITTER', 'INSTAGRAM', 'SNAPCHAT',
       'WHATSAPP', 'LINKEDIN', 'PINTEREST', 'TIKTOK', 'BEREAL', 'REDDIT']

df = pd.DataFrame(columns=['Group','YEAR','PercentNegDiff'])
for group in groups:
    new_df = get_percent_increase(group,True)
    new_df = new_df[(new_df[group] != 'nan') | (new_df[group] != 'Refused')]
    new_df[group] = group + ': ' + new_df[group]
    cols = list(new_df.columns)
    cols[0] = 'Group'
    new_df.columns = cols
    df = pd.concat([df,new_df])

def create_graph(df,year,topN,asc):
    df = df[df['YEAR'] == year].sort_values('PercentNegDiff',ascending=asc)
    df = df.iloc[0:topN]
    write_to_file(df)

    fig = px.bar(
        df,
        y='Group',
        x='PercentNegDiff',
        barmode='group',
        # color=df.columns[0],
        hover_data=['YEAR'],
        category_orders=category_orders,
        color_discrete_map=color_map,
        orientation="h"
    )
    # type='category' force x-axis to be discrete categories, not continuous bins so years won't be combined
    fig.update_yaxes(title="Percentage").show()

create_graph(df,'2022',15,False)